# Эксперименты E1 и E2: обнаружение СИЗ в Google Colab

Запускайте ячейки сверху вниз. В меню **Среда выполнения → Сменить среду выполнения** выберите GPU. Набор ожидается в `Мой диск/lab_3_ai/data`. Поддерживаются папки `raw/` вместе с `split_manifest.csv`, готовая `processed/` или архив `data.zip`/`dataset.zip` с таким же содержимым. Для сравнения используется закреплённое разбиение проекта: 1961/420/420 изображений. Результаты обучения и оценки сохраняются в `Мой диск/lab_3_ai/runs/ppe/`.

E1: YOLOv8n, 25 эпох, `imgsz=640`, `batch=16`. E2: те же веса, данные, число эпох и seed, `imgsz=960`, `batch=4`. Пакет E2 уменьшен для памяти GPU, как в `experiments/e2.py`, поэтому изменяется не только разрешение.


In [ ]:
%pip -q install 'ultralytics==8.4.191'


ERROR: Operation cancelled by user


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Подготовка данных

Архив, если он есть, распаковывается в `/content`; данные из папки на Google Drive копируются туда же. Это ускоряет чтение при обучении. Исходные файлы на Drive не изменяются. Файл `split_manifest.csv` сохраняет то же разбиение, что применялось в локальных E1 и E2.


In [ ]:
import csv
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import torch
import yaml

DRIVE_ROOT = Path('/content/drive/MyDrive/lab_3_ai')
DRIVE_DATA = DRIVE_ROOT / 'data'
WORK_ROOT = Path('/content/ppe_colab')
DATA_ROOT = WORK_ROOT / 'data'
RUNS_ROOT = DRIVE_ROOT / 'runs' / 'ppe'

assert torch.cuda.is_available(), 'Выберите GPU в настройках среды выполнения Colab.'
assert DRIVE_DATA.exists(), f'Не найдена папка на Google Drive: {DRIVE_DATA}'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

archive = next((p for p in (DRIVE_DATA / 'data.zip', DRIVE_DATA / 'dataset.zip',
                            DRIVE_ROOT / 'data.zip', DRIVE_ROOT / 'dataset.zip') if p.is_file()), None)
if archive:
    print('Распаковка', archive)
    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            target = (WORK_ROOT / member.filename).resolve()
            if not target.is_relative_to(WORK_ROOT.resolve()):
                raise ValueError(f'Недопустимый путь в ZIP: {member.filename}')
        zf.extractall(WORK_ROOT)
    if not DATA_ROOT.exists():
        raise FileNotFoundError('В архиве ожидается папка data/.')
else:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    # Манифест и исходные изображения нужны для воспроизведения закреплённого разбиения.
    for filename in ('split_manifest.csv',):
        source = DRIVE_DATA / filename
        if source.is_file():
            shutil.copy2(source, DATA_ROOT / filename)
    has_raw = (DRIVE_DATA / 'raw').is_dir() and (DATA_ROOT / 'split_manifest.csv').is_file()
    has_processed = (DRIVE_DATA / 'processed').is_dir()
    if has_raw:
        print('Копирование исходных изображений и меток с Google Drive...')
        shutil.copytree(DRIVE_DATA / 'raw', DATA_ROOT / 'raw', dirs_exist_ok=True)
    if has_processed:
        print('Копирование готового разбиения с Google Drive...')
        shutil.copytree(DRIVE_DATA / 'processed', DATA_ROOT / 'processed', dirs_exist_ok=True)
    if not (has_raw or has_processed):
        raise FileNotFoundError(
            f'В {DRIVE_DATA} нужны raw/ + split_manifest.csv, processed/ или data.zip/dataset.zip.'
        )

print('Данные:', DATA_ROOT)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Воссоздаём processed/ по закреплённому манифесту; symlink из локального проекта не нужны.
SPLIT_NAMES = {'обучение': 'train', 'проверка': 'val', 'тест': 'test'}
expected = {'train': 1961, 'val': 420, 'test': 420}
manifest = DATA_ROOT / 'split_manifest.csv'
processed = DATA_ROOT / 'processed'
counts = Counter()

if manifest.is_file() and (DATA_ROOT / 'raw').is_dir():
    with manifest.open(newline='', encoding='utf-8') as stream:
        for row in csv.DictReader(stream):
            split = SPLIT_NAMES[row['новая_часть']]
            for column, kind in (('изображение', 'images'), ('разметка', 'labels')):
                relative = Path(row[column])
                if relative.parts[0] != 'data' or '..' in relative.parts:
                    raise ValueError(f'Некорректный путь в манифесте: {relative}')
                source = DATA_ROOT.joinpath(*relative.parts[1:])
                target = processed / split / kind / source.name
                target.parent.mkdir(parents=True, exist_ok=True)
                if not target.is_file():
                    if source.is_file():
                        shutil.copy2(source, target)
                    else:
                        raise FileNotFoundError(
                            f'Отсутствует {source} и готовая копия {target}. '
                            f'Проверьте загрузку датасета на Google Drive; '
                            f'можно положить полный data.zip или dataset.zip в {DRIVE_DATA}.'
                        )
            counts[split] += 1
else:
    for split in expected:
        counts[split] = len(list((processed / split / 'images').glob('*.jpg')))

if dict(counts) != expected:
    raise ValueError(f'Неверный состав разбиения: {dict(counts)}; ожидается {expected}')
for split in expected:
    images = list((processed / split / 'images').glob('*.jpg'))
    if any(not (processed / split / 'labels' / f'{image.stem}.txt').is_file() for image in images):
        raise FileNotFoundError(f'В {split} есть изображения без YOLO-разметки')
print('Разбиение:', dict(counts))

names = ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest',
         'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']
config = {'path': str(processed), 'train': 'train/images', 'val': 'val/images',
          'test': 'test/images', 'names': dict(enumerate(names))}
DATA_YAML = WORK_ROOT / 'ppe_colab.yaml'
DATA_YAML.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('Конфигурация:', DATA_YAML)


## E1: базовое обучение, 640 px

Начальные веса `yolov8n.pt` скачиваются Ultralytics при первом запуске. После обучения лучшие веса оцениваются на **тестовой** части. Каждому запуску создаётся отдельная папка на Google Drive.


In [ ]:
import json
from ultralytics import YOLO


def run_experiment(name: str, imgsz: int, train_batch: int):
    model = YOLO('yolov8n.pt')
    model.train(
        data=str(DATA_YAML), epochs=25, imgsz=imgsz, batch=train_batch,
        seed=42, deterministic=True, device=0,
        project=str(RUNS_ROOT), name=name,
    )
    run_dir = Path(model.trainer.save_dir)
    best = run_dir / 'weights' / 'best.pt'
    if not best.is_file():
        raise FileNotFoundError(best)
    metrics = YOLO(str(best)).val(
        data=str(DATA_YAML), split='test', imgsz=imgsz, batch=16,
        device=0, plots=True, project=str(run_dir), name='test',
    )
    output = Path(metrics.save_dir)
    (output / 'metrics.json').write_text(
        json.dumps(metrics.results_dict, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    print(f'{name}: mAP@0.5={metrics.box.map50:.4f}; '
          f'mAP@0.5:0.95={metrics.box.map:.4f}')
    print('Веса и журнал:', run_dir)
    print('Тестовые метрики:', output)
    return metrics

metrics_e1 = run_experiment('e1-opora-colab', imgsz=640, train_batch=16)


## E2: увеличение разрешения до 960 px

E2 начинается заново с исходных предобученных весов, а не с весов E1. При нехватке видеопамяти проверьте выделенную GPU; изменение параметров повлияет на сопоставимость с проектным E2.


In [ ]:
metrics_e2 = run_experiment('e2-img960-colab', imgsz=960, train_batch=4)


## Сравнение тестовых метрик

Оценка здесь выполняется один раз после каждого обучения. Для подбора гиперпараметров используйте валидационную часть, не тестовую.


In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {'experiment': 'E1', 'imgsz': 640, 'train_batch': 16,
     'mAP50': metrics_e1.box.map50, 'mAP50-95': metrics_e1.box.map},
    {'experiment': 'E2', 'imgsz': 960, 'train_batch': 4,
     'mAP50': metrics_e2.box.map50, 'mAP50-95': metrics_e2.box.map},
])
display(comparison)
comparison.to_csv(RUNS_ROOT / 'e1_e2_colab_comparison.csv', index=False)
